<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/03_대규모_언어_모델_자세히_살펴보기.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 3 서론](#chapter3)
* [Chapter 3-1 트랜스포머 모델 개요](#chapter3-1)
    * [Chapter 3-1-1 훈련된 트랜스포머 LLM의 입력과 출력](#chapter3-1-1)
    * [Chapter 3-1-2 정방향 계산의 구성 요소](#chapter3-1-2)
    * [Chapter 3-1-3 확률 분포로부터 하나의 토큰 선택히기(샘플링/디코딩)](#chapter3-1-3)
    * [Chapter 3-1-4 병렬처리 토큰 처리와 문맥크기](#chapter3-1-4)
    * [Chapter 3-1-5 키와 값을 캐싱하여 생성 속도 높이기](#chapter3-1-5)
    * [Chapter 3-1-6 트랜스포머 블록 내부](#chapter3-1-6)
* [Chapter 3-2 트랜스포머 아키넥터의 최근 발전 사항](#chapter3-2)   
    * [Chapter 3-2-1 효율적인 어텐션](#chapter3-2-1)   
    * [Chapter 3-2-2 트랜스포머 블록](#chapter3-2-2)   
    * [Chapter 3-2-3 위치 임베딩(RoPE)](#chapter3-2-3)   

## Chapter 3 서론 <a class="anchor" id="chapter3"></a>
1. 이 장에서 트랜스포머 언어 모델의 작동 방식을 직관적으로 이해해보도록한다.
    - 텍스트 생성 모델이 주요 관심사이므로 특별히 생성형 LLM을 자세히 살펴보도록 한다.

2. 언어 모델을 로드하고 파이프라인 객체를 만들어 텍스트 생성을 위한 준비를 한다.
    - 코드보다는 관려된 개념을 이해하는 데 초점을 맞춘다.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

#토크나이져를 로드한다.
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")


In [2]:
# 모델을 로드한다.
model = AutoModelForCausalLM.from_pretrained(
	"microsoft/Phi-3-mini-4k-instruct",
	device_map="cuda",
	dtype="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [3]:
# 파이프라인 객체를 만든다.
generator = pipeline("text-generation",
                     model=model,
                     tokenizer=tokenizer,
                     return_full_text=False,
                     max_new_tokens=100,
                     do_sample=False
                    )

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Chapter 3-1 트랜스포머 모델 개요 <a class="anchor" id="chapter3-1"></a>
#### Chapter 3-1-1 훈련된 트랜스포머 LLM의 입력과 출력 <a class="anchor" id="chapter3-1-1"></a>
1. 트랜스포머 LLM의 동작 방식을 이해하는 가장 일반적인 방법은 모델을 하나의 소프트웨어 시스템으로 생각하는 것이다.
    - 이 시스템은 텍스트를 입력 받아 응답으로 텍스트를 생성할 수 있다.

2. 아래의 그림은 이메일을 작성할 수 있는 모델의 예를 보여준다.

     ![모델 예시](./image/03_model_example1.png)


3. 이 모델은 한 번에 모든 텍스트를 생성하지 않고 한 번에 하나의 토큰씩 생성한다.
    - 아래의 이미지는 응답으로 네 개의 토큰을 생성하는 과정을 보여준다.
    - 각각의 토큰 생성 과정은 모델을 통과하는 한 번의 정방향계산(forward pass)을 필요로 한다.

        ![토큰 생성 과정](./image/03_token_generation.png)

4. 하나의 토큰을 생성한 후에, 출력 토큰을 입력 프롬프트의 끝에 추가하여 다음 생선 단계를 위한 프롬프트로 사용한다.
    - 출력 토큰이 추가된 프롬프트가 모델에 전달되어 새로운 텍스트를 생성하기 위한 정방향 계산이 수행된다.

        ![프롬프트 업데이트](./image/03_prompt_update.png)

5. 모델은 입력 프롬프트를 기반으로 다음 토큰을 예측하는 작업을 반복하여 텍스트를 생성한다.
    - 이렇게 이전 예측을 사용해 다음 예측을 만드는 모델을 자기회귀 모델(autoregressive model)이라고 한다.

6. 텍스트 생성 LLM은 이 같은 특성이 있어 자기회귀 모델이라고 한다.
    - 이 특성은 자기회귀적이지 않은 BERT와 같은 모델과 구별되는 중요한 특징이다.

In [7]:
# LLM은 한 토큰씩 생성하는 자귀회귀적인 과정을 내부에서 수행한다.
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Mention that you've already taken steps to prevent it in the future.


Email to Sarah:

Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have been helping you with your gardening for some time now


#### Chapter 3-1-2 정방향 계산의 구성 요소 <a class="anchor" id="chapter3-1-2"></a>
1. 토크나이저와 언어 모델링 헤드(LM 헤드)는 중용한 구성요소이다.
    - 아래의 그림에서 두 구성요소가 전체 시스템에서 어디에 위치하는지 알 수있다.
    - 토크나이저가 텍스트를 토큰 시퀸스로 변환하고, 트랜스포머 블럭을 통과한 출력 정보를 LM 헤드가 다음 토큰에 대한 확률로 변환한다.

        ![구성 요소](./image/03_components.png)

2. 토크나이저는 토큰의 테이블인 어휘사전을 포함하고 있고, 트랜스포머 모델은 어휘사전에 있는 가 토큰에 연관된 벡터 표현인 토큰 임베딩을 가지고 있다.
    - 50,00개의 토큰으로 구성된 어휘사전과 모델에 있는 임베딩을 그림으로 표현하면 아래와 같다.
    
        ![토크나이저와 임베딩](./image/03_tokenizer_embedding.png)

3. 토큰 하나를 생성하기 위해 토크나이저 -> 트랜스포머 블록 -> LM 헤드를 통과한 후 토큰에 대한 확률을 출력한다.
    - 정방향 계산의 마지막에 모델은 어휘사전에 있는 모든 토큰에 대한 확률 점수를 출력한다.
    
        ![토큰 생성 과정](./image/03_token_generation_process.png)

4. 다양한 종류의 시스템을 만들기 위해 트랜스포머 블록의 스텍에 여러 종류의 헤드를 붙일 수 있다.
    - LM 헤드는 외에 분류 헤드, 회귀 헤드, 시퀀스 레이블링 헤드 등을 붙일 수 있다.

In [ ]:
# model을 출력하여 세부 정보를 확인한다.
#   - 모델 층이 여러 단계로 중첩되어 있으면 가장 상위 층은 model과 lm_head 이다.
#   - 모델의 임베딩 토큰은 32,064개이 이고, 벡터의 크기는 3,072이다.
#   - 가각의 트랜스포머 블록 안에는 어텐션 층과 피드포워드 신경만이 있다.
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

#### Chapter 3-1-3 확률 분포로부터 하나의 토큰 선택히기(샘플링/디코딩) <a class="anchor" id="chapter3-1-3"></a>
1. 토큰 생성 과정의 마지막에 모델이 출력하는 것은 어휘사전에 있는 모든 토큰에 대한 확률점수 이다.
    - 이 확률 분포에서 하나의 토큰을 선택하는 방법을 디코딩(decoding) 또는 샘플링(sampling)이라고 한다.
    - 아래의 그림은 토큰 'Dear'를 선택하는 방법을 보여준다.

        ![디코딩](./image/03_decoding.png)

2. 가장 간단한 디코딩 전략은 확률점수가 가장 높은 토큰을 고르는 것이다.
    - 이 전략은 탐욕적 디코딩(greedy decoding)이라고 부리는데 생성된 텍스트가 항상 동일하게 된다는 단점이 있다.
    - 약간의 무작위성을 부여하거나 이따금 두 번째나 세 번째로 높은 점수를 가진 토큰을 선택하는 전략이 더 다양하고 흥미로운 텍스트를 생성할 수 있다.

3. 토큰 'Dear'가 다음 토큰이 될 확률을 40%라고 하면, 다른 토큰들도 각각의 점수에 따라 선택될 가능성이 있다.

In [6]:
prompt = "The capital of France is"

# 입력 프롬프트를 토큰화한다.
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# lm_head 앞에 있는 model의 출력을 얻는다.
#   -  output_hidden_states=True로 설정하면 모델이 각 층의 출력을 반환한다.
model_output = model.model(input_ids)
print(model_output)

# lm_head의 출력을 얻는다.
lm_head_output = model.lm_head(model_output[0])
print(lm_head_output.shape)

BaseModelOutputWithPast(last_hidden_state=tensor([[[-0.2969,  1.1484,  0.3145,  ..., -0.3125,  0.7070,  0.1396],
         [-0.1357,  0.3164,  0.3848,  ...,  0.5469, -0.1406, -0.6094],
         [-0.5781,  1.0625,  1.5547,  ..., -0.4180,  0.2773,  0.3945],
         [-0.3984,  0.6836,  0.2500,  ..., -0.0635,  0.8047, -0.6680],
         [-0.6641,  0.6953,  0.5391,  ...,  0.2559,  0.1738, -0.2871]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<MulBackward0>), past_key_values=DynamicCache(layers=[DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, Dynamic

In [7]:
# lm_head 출력의 마지막 토큰의 확률 점수를 얻는다.
last_token_logits = lm_head_output[:, -1, :]
print(last_token_logits)

tensor([[27.7500, 29.3750, 28.0000,  ..., 20.3750, 20.3750, 20.3750]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [8]:
# 가장 큰 확률 점수를 가진 토큰의 인덱스를 얻는다.
token_id = last_token_logits.argmax(-1)
print(token_id)

tensor([3681], device='cuda:0')


In [9]:
# decoder 토크나이저를 사용하여 토큰 ID를 텍스트로 변환한다.
generated_token  = tokenizer.decode(token_id)
print(generated_token)

Paris


#### Chapter 3-1-4 병렬처리 토큰 처리와 문맥크기 <a class="anchor" id="chapter3-1-4"></a>
1. 트랜스포머는 언어 처리 분양의 기존 신경망 구조보다 병렬 처리에 뛰어나다.
    - 각 토큰은 독자적인 계산 스티림을 통해 처리된다.
    - 트랜스포머 모델은 동시에 처리한 수 있는 토큰의 수에 제한이 있으며, 이 제한을 문맥 길이라고 부른다.

        ![병렬 처리](./image/03_parallel_processing.png)

2. 각 토큰 스트림은 입력 벡터로 시작한다.
    - 입력 벡터는 임베딩 벡터와 위치 임베딩을 더한 값이다.
    - 아래의 그림과 같이 각 처리 스트림은 입력 벡터를 받고 동일 크기의 결과 벡터를 생성한다.

        ![입력 벡터](./image/03_input_vector2.png)

3. 텍스트 생성의 경우 마지막 스트림만 출력 결과만 사용해 다음 토큰을 예측하고, 이 출력 벡터만 LM 헤드로 전달된다.

4. 마지막 토큰을 제외하고 나모지 모든 토큰을 버리는데 이것을 계산하는 이유는 이전 스트림의 계산이 최종 스트림 계산에 필요하기 때문이다. 
    - 트랜스포머 모델의 어텐션 메커니즘에서 이전 스트림의 출력을 사용한다.

5. 앞의 예제에서 lm_head의 출력이 [1, 5, 32064]인 것은 입력의 크기가 [1, 5, 3072]이기 대문이다.
    - 1은 배치 크기, 5는 토큰의 수, 3072는 트랜스포머 블록의 출력 벡터의 크기이다.
    - LM 헤드는 이 벡터를 어휘사전에 있는 모든 토큰에 대한 점수로 변환하는 선형 레이어이므로 출력의 크기는 [1, 5, 32064]가 된다.

#### Chapter 3-1-5 키와 값을 캐싱하여 생성 속도 높이기 <a class="anchor" id="chapter3-1-5"></a>
1. 두 번째 토큰을 생성할 때 단순히 첫 번째 출력 토큰을 입력에 덧붙여 또 한 번 모델의 정발행 계산을 수행한다.
    - 모델이 이전 계산 결과를 캐싱할 수 있다면 이전 스트림의 계산을 반복할 필요가 없다.
    - 즉 마지막 스트림에 대한 계산만 필요하다.
    
        ![키-값 캐싱](./image/03_key_value_caching.png)

2. 이런 최적화 기법은 키와 값 캐쉬라고 부르며 텍스트 생성 과정과 속도를 크게 높인다.
    - 훈련 시에는 자기회귀적으로 텍스트를 이어가지 않고 문맥 길이의 샘플을 한번에 처리하므로 키-값 캐싱을 사용하지 않는다.

3. 허깅 페이스의 transformers는 기본적으로 캐싱을 사용하며, use_cache=False로 설정하여 캐싱을 비활성화할 수 있다.
        

In [4]:
# use_cache 설정에 따른 생성 속도 비교
prompt = "Write a very long email apologize to Sarah for the tragtic gradening mishape. Explain hot it happoened"

# 입력 프롬프트 토큰화
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

In [6]:
%%timeit -n 1

# 텍스트를 생성한다.
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


2.41 s ± 418 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
%%timeit -n 1

# 텍스트를 생성한다.
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=False)

2.54 s ± 342 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


#### Chapter 3-1-6 트랜스포머 블록 내부  <a class="anchor" id="chapter3-1-6"></a>
1. 트랜스포머 LLM은 일련의 트랜스포머 블록으로 구성되며 각 블록은 입력을 처리한 다음 출력을 다음 블록으로 전달한다.
    - 트랜스포머 블록은 입력 벡터를 받아서 동일한 크기의 출력 벡터를 생성하는 함수로 생각할 수 있다.
    - 트랜스포머 LLM 계산 과정의 대부분은 트랜스포머 블럭 안에서 일어난다.

        ![트랜스포머 블록](./image/03_transformer_block.png)


2. 트랜스포머 블록은 두 개의 구성 요소가 있다.
    - 어텐션 층: 입력 벡터의 각 요소가 다른 요소에 주의를 기울이는 메커니즘을 구현한다.
    - 피드포워드 네트워크: 어텐션 층의 출력을 처리하여 다음 블록으로 전달할 벡터를 생성한다.

        ![트랜스포머 블록 구성 요소](./image/03_transformer_block_components.png)

3. 피드포워드 신경망을 직관적으로 이해하기 위해 간단한 예를 생각해보자
    - "The Shawshank"를 언어 모델에 입력하면 가장 높은 확률로 다음 토큰이 영화제목인 "Redemption"이 될 것이다.
    - 모델이 "The Shawshank Redemption"이 여러번 언급된 대규모 텍스트 데이터에서 성공적으로 훈련될 때 이 작업을 성공시키기 위한 정보를 학습하고 저장한다.
    - 트랜스포머 블록의 피드포워드 신경망은 모델의 기억과 보간의 대부분을 담당한다.

4. LLM이 성공적으로 훈련되려면 많은 정보를 암기해야한다.
    - 단순히 대용량 데이터베이스이면 되는 것이아니며, 암기는 텍스트 생성 레시피에 필요한 하나의 요소일 뿐이다.
    - 모델은 데이터 포인트 사이와 복잡한 패턴 사이를 보간하여 일반화할 수 있다.
        - 보간이란 모델이 훈련 데이터셋에 없는 입력에 대해 훈련 데이터셋에 있는 입력과 유사한 패턴을 인식하여 적절한 출력을 생성하는 능력이다.
    - 즉 과거에 본 적 없고 훈련 데이터셋에 없는 입력에 대해 잘 수행된다는 의미이다.


5. 어텐션은 모델이 특정 토큰을 처리할 때 문맥을 통합하도록 도와주는 메거니즘이다.
    - 어텐션은 모델이 입력 시퀸스의 다른 부분에 주의를 기울이는 방법을 제공한다.
    - 예를 들어 "The dog chased the squirrel because it"라는 문장에서 모델이 'it' 다음에 올 토큰을 예측하기위해 'it'이 무엇을 가리키는지 알아야한다.
    - 어텐션은 모델이 'it'이 'the dog'를 가리키는지 'the squirrel'을 가리키는지 결정하는 데 도움이 된다.

6. 어텐션은 문맥 정보를 'it' 토큰의 표현에 추가한다.
    - 모델은 훈련 데이터셋에서 보고 배운 패턴을 기반으로 이를 수행한다.
    - 셀프 어텐션 층은 현재 토큰을 처리하는 데 도움이 되는 이전 위치의 정보를 통합한다.

        ![어텐션](./image/03_attention.png)

7. 어텐션 메커니즘을 간단하게 표현하면 아래의 그림과 같다.
    - 현재 처리대상 토큰에 대해서 어텐션 메커니즘이 작동한다.
    - 해당 위치의 벡터와 어텐션 메커니즘에 따라 이전 원소에서 얻은 정보를 통하아여 새로운 벡터를 생성한다.

        ![어텐션 메커니즘](./image/03_attention_mechanism.png)

8. 어텐션 메커니즘의 주요한 두 개의 단계는 다음과 같다.
    - 이전 입력 토큰이 현재 터리 대상 토큰에 얼마나 연관이 있는지 점수를 매긴다.
    - 다양한 위치에서 얻은 이 점수를 하나의 출력 벡터에 통합한다.
    
        ![어텐션 메커니즘 단계](./image/03_attention_mechanism_steps.png)

9. 트랜스포머에 더 광범위한 어텐션 능력을 부여하기 위해 어텐션 메커니즘을 여러 벌 만들어 병렬로 수행하며, 이를 어텐션 헤드라고 부른다.
    - 이는 입력 시퀸스에서 한 번에 여러 패턴에 주의를 기울여야 하는 복잡한 패턴을 모델링하기 위한 모델의 용량을 증가시킨다.
    - 여러 개의 어텐션을 병렬로 계산하여 LLM의 성능을 높인다.

        ![어텐션 헤드](./image/03_attention_heads.png)

10. 하나의 어텐션 헤드에서 어텐션 점수가 계산되는 과정을 알아보기 전에 다음의 내용을 숙지한다.
    - 생성 LLM의 어텐션 층은 한 위치의 토큰에 대한 어텐션을 처리한다.
        - 이 층의 입력은 현재 위치 또는 토큰에 대한 벡터 표현이다.
    - 이전 토큰에서 과련 정보를 통합하여 현재 위치에 대한 새로운 표현을 생성하는 것이 목표다.
        - "Sarah fed the cat because it"이란 문장의 마지막 토큰을 처리한다면 'it'이 'cat'을 나타내야한다.
        - 어텐션은 'cat' 토큰에서 고양이에 대한 정보를 추출하여 통합해야 한다.
    - 훈련 과정을 통해 이 계산에 활용되는 세 개의 투영 행렬이 만들어진다.
        - 쿼리 행렬, 키 행렬, 값 행렬이라고 불리는 이 행렬은 어텐션 계산에서 중요한 역할을 한다.  
   
        ![어텐션 계산](./image/03_attention_calculation.png)

11. 입력을 각각의 투영 행렬과 곱해서 세 개의 새로운 행렬을 만드는 것으로 시작한다.
    - 일반적으로 입력을 세 개의 밀집층에 통화시켜 행렬을 만든다.

12. 이 정보는 두 개의 어텐션 단계를 수행하는데 도움이된다.
    - 관려성 점수 계산, 정보 통합

13. 각 행렬에서 맨 아래 행이 현재 위치의 토큰에 해당된다.
    - 층의 입력과 각각의 투영 행렬을 곱하여 쿼리, 키 행렬을 만든다.

        ![어텐션 계산 행렬](./image/03_attention_calculation_matrices.png)

14. 생성형 트랜스포머에서는 한 번에 하나의 토큰을 생성한다.
    - 언텐션 메커니즘은 한 위치에만 관심을 두며 다른 위치에서 정보를 가져와 어떻게 이 위치에 반영할 수 있는지 관심을 둔다.

15. 언텐션 관련성 점수 계산은 현재 위치의 쿼리 벡터와 키 행렬을 곱하여 수행된다.
    - 이를 통해 이전 토큰이 얼마나 관련 있는지를 나타내는 점수가 만들어 진다.
    - 그다음 소프트맥스 연산을 통해 점수의 합이 1이 되도록 정규화한다.

        ![어텐션 점수 계산](./image/03_attention_score_calculation.png)


16. 관련성 점수가 준비되면 각 토큰에 연관된 값 벡터와 이 점수를 곱한다.
    - 그런 다음 이 가중치가 적용된 값 벡터를 모두 더하여 현재 위치에 대한 새로운 표현을 만든다.
    - 이 새로운 벡터는 트랜스포머 블록의 다음 단계로 전달된다.

        ![어텐션 정보 통합](./image/03_attention_information_integration.png)

### Chapter 3-2 트랜스포머 아키넥터의 최근 발전 사항 <a class="anchor" id="chapter3-2"></a>
#### Chapter 3-2-1 효율적인 어텐션 <a class="anchor" id="chapter3-2-1"></a>
1. 트랜스포머의 어텐션 층이 계산 비용이 가장 비싼 부분이기 때문에 기술 발전에서 중용한 대상이 되었다.

2. 트랜서포머 모델이 점차 커지면서 희소 어텐션과 슬라이딩 윈도 어텐션과 같은 아이디어가 어텐션 계산의 효율성을 개선하였다.
    - 희소 어텐션은 모델이 주의를 기울일 수 있는 이전 토큰의 문맥을 제한한다.
    - 로컬 어텐션의 성능을 높이기 위해 적은 수의 이전 위치에만 주의를 기울인다.
    
        ![효율적인 어텐션](./image/03_efficient_attention.png)

3. 모든 트랜스포머 블록에서 사용하는 것은 아니다.
    - 모델이 이전 토큰 중 일부만 볼 수 있다면 생성 텍스트의 품질이 저하될 수 있기 때문이다.
    - GPT-3은 완전 어텐션과 효율적인 어텐션을 섞어 사용한다.
    - 완전 어텐션과 희소 어텐션을 이미지로 표현하면 아래와 같다.
    - 진한 파란색 토큰을 처리할 때 주의를 기울일 수 있는 토큰색을 아래의 그림에서 볼 수 있다.   

        ![완전 어텐션과 희소 어텐션](./image/03_full_sparse_attention3.png)


4. 멀티 쿼리 어텐션과 그룹 쿼리 어텐션
    - 멀티 쿼리 어텐션: 각 어텐션 헤드가 동일한 키와 값 행렬을 공유하는 어텐션 메커니즘이다.
    - 그룹 쿼리 어텐션: 어텐션 헤드를 여러 그룹으로 나누고 각 그룹이 자체 키와 값 행렬을 사용하는 어텐션 메커니즘이다.
    
        ![멀티 쿼리 어텐션과 그룹 쿼리 어텐션](./image/03_multi_group_query_attention.png)

5. 멀티 헤드 어텐션에서는 각각의 헤드가 주어진 입력으로 계산한 독자적인 쿼리, 키, 값 행렬을 사용한다.

    ![멀티 헤드 어텐션](./image/03_multi_head_attention.png)

5. 멀티 쿼리 어텐션의 최적화 방법은 키와 값 행렬을 모든 헤드에서 공유하는 것이다.
    - 이렇게 하면 모델이 각 헤드에 대해 별도의 키와 값 행렬을 계산할 필요가 없으므로 어텐션 계산이 더 효율적이 된다.

        ![멀티 쿼리 어텐션](./image/03_multi_query_attention.png)

6. 모델의 크기가 커지면서 이런 최적화는 매우 과할 수 있으며 모델의 품질을 향상시키기 위해 메모리를 조금더 사용할 수 있다.
    - 이것이 그룹 쿼리 어텐션이 등장한 이유이다.
    - 키 값 행렬을 한 개만 사용하는 것이 아니라 더 많이 사용할 수 있다. 
    - 그룹 쿼리 어텐션은 멀티 쿼리 어텐션의 효율성을 조금 희생하여 그룹마다 키/값 행렬을 공유하게 함으로써 품질을 크게 개선한다.
    - 각 글룹으 여러개의 어텐션 헤드로 구성된다.

        ![그룹 쿼리 어텐션](./image/03_group_query_attention.png)

7. 플래쉬 어텐션은 트랜스포머 LLM이 GPU에서 훈련하거나 추론할 때 속도를 크게 높일 수 있는 효율적인 어텐션 구현이다.
    - GPU의 공유 메모리와(SRAM)과 고대역 폭 메모리(HBM) 사이에 이동하는 값을 최적화하여 어텐션 계산 속도를 높입니다.

#### Chapter 3-2-2 트랜스포머 블록 <a class="anchor" id="chapter3-2-2"></a>
1. 트랜슾포머 블록의 주요 구성 요소 두 가지는 어텐션층과 피드포워드 신경망이다.
    - 아래의 이미지로 표현 할 수 있으며 잔차 연결과 정규화 층도 포함되어 있다.

        ![트랜스포머 블록 구성 요소](./image/03_transformer_block_components2.png)

2. 최신 트랜스포머 모델에서는 몇가지 개선점이 있다.
    - 정규화가 어텐션과 피드포워드 신경망 이전에 등장한다.
        - 이 방식이 훈련 시간을 단축시켜 준다고 보고되있다.
    - 정규화보다 더 간단한 고 효율적인 RMSNorm을 사용한다.
    - ReLU 활성화 함수 대신 SwiGLU 같은 새로운 활성화 함수를 사용한다.
    
        ![트랜스포머 블록 개선점](./image/03_transformer_block_improvements.png)

#### Chapter 3-2-3 위치 임베딩(RoPE) <a class="anchor" id="chapter3-2-3"></a>
1. 위치 임베딩은 모델이 시퀸스 문장 안에서 각 토큰의 순서를 이해할 수 있도록 하는 메커니즘이다.
    - 로터리 위치 임베딩이 가장 널리 사용되는 위치 임베딩 방법 중 하나이다.

2. 원본 트랜스포머에서는 절대 위치 임베딩을 사용했다.
    - 첫 번째 토큰의 위치는 1, 두 번째 토큰의 위치는 2, 세 번째 토큰의 위치는 3과 같이 각 토큰에 고유한 위치 벡터를 할당한다.
    - 훈련 세트에 있는 많은 문서가 문맥보다 짧다. 10개의 단어로 이루어진 문장에 4K 문맥 전체를 할당하는 것은 비효율적이다.
    - 아래의 그림과 같이 데이터 패킹을 통해 문서를 문맥에 맞게 효율적으로 배치할 수 있다.

        ![데이터 패킹](./image/03_data_packing.png)

3. 데이터 패킹과 그 외 실용적인 고려 사항에 맞추어 위치 임베딩을 적용해야한다.
    - 예를 들어 문서 50이 위치 50부터 시작한다고 가정하고, 이 문서의 첫 번째 토큰의 위치가 50이라고 모델에게 알려주면 성능에 영향을 미칠 수 있다.


4. 로터리 임베딩은 상대적인 토큰 위치 정보를 인코딩하는 방법으로, 임베딩 공간에서 벡터를 회전시키는 아이디어를 기반으로 한다.
    - 회전된 두 벡터를 점곱하면 그 결과에서 두 벡터 사이의 상대적인 위치정보가 표현된다.
    - 아래의 글림과 같이 정방향 계산의 어텐션 단계에서 로터리 임베딩이 추가된다.

        ![로터리 임베딩](./image/03_rotary_embedding.png)

5. 어텐션 과정동안 위치 정보가 관련성 점수를 곗나하기 위해 쿼리, 키, 행렬을 곱하기 전에 특별한 방식으로 혼합된다.
    - 로터리 위치 임베딩은 셀프 어텐션 관련성 점수 계산 단계 직전에 토큰 표현에 추가된다.

        ![로터리 임베딩 계산](./image/03_rotary_embedding_calculation2.png)